# H&M 60일 single-adapter screening (seed 42, validation-only)

이 노트북은 H&M 최근 60일 창에서 single-adapter 파이프라인과 비용을 점검하는 탐색용 screening입니다. 본실험(약 2년)을 대체하지 않으며, test와 holdout 정답을 열지 않습니다.

고정 분할의 validation/test/holdout 예약기간은 각각 7일이므로 실제 학습기간은 약 39일입니다. 행동 encoder는 14일 관측에서 7일 미래를 예측하고, train 안에서 anchor `(21, 14, 7)`을 사용합니다 (`14 + 21 = 35`).

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REVIEWED_SHA = '95cc4ea3b24f13581f215e595b0d8bbdf6b93338'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-hm-w60')
assert not REPO_DIR.exists(), (
    f'{REPO_DIR} already exists. Start a fresh Colab runtime before running this notebook.'
)
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('검토된 소스:', actual_sha)

In [ ]:
import json
import torch

from lightgcn_clv_single import (
    HM_60DAY_FROZEN,
    configure_hm_60day_run,
    preflight_summary,
    run_experiment,
)

RESULT_ROOT = Path('/content/drive/MyDrive/논문/data')
SINGLE_OUT_DIR = RESULT_ROOT / 'results_clv_single_hm_w60'
M1_CHECKPOINT_DIR = RESULT_ROOT / 'results_v3_hm_w60'
cfg = configure_hm_60day_run(
    out_dir=str(SINGLE_OUT_DIR),
    m1_checkpoint_dir=str(M1_CHECKPOINT_DIR),
)

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))
assert cfg.dataset == 'hm'
assert cfg.window_days == 60
assert cfg.input_days == 14
assert cfg.target_days == 7
assert cfg.anchor_offsets == (21, 14, 7)
assert cfg.eval_test is False and cfg.eval_holdout is False
for setting, expected in HM_60DAY_FROZEN.items():
    assert getattr(cfg, setting) == expected, (setting, getattr(cfg, setting), expected)
assert Path(cfg.out_dir) == SINGLE_OUT_DIR
assert Path(cfg.m1_checkpoint_dir) == M1_CHECKPOINT_DIR

In [ ]:
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['estimated_train_days'] == 39
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
print('아직 데이터 전처리나 모델 학습은 시작되지 않았습니다.')

In [ ]:
ACKNOWLEDGE_HIGH_COST = False
assert ACKNOWLEDGE_HIGH_COST, (
    '설정과 예상 비용을 검토한 뒤 ACKNOWLEDGE_HIGH_COST=True로 바꾸세요.'
)

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

print('절대 성능 곡선 (validation):')
display(result_df.sort_values(['model_id', 'split', 'lambda']))
absolute_curves = result_df[result_df['split'].eq('val')].pivot_table(
    index='lambda', columns='model_id', values='revenue@10', aggfunc='first'
)
absolute_curves.plot(marker='o', title='Validation weighted-hit@10 absolute curves')
plt.xlabel('lambda')
plt.ylabel('price/purchase-amount weighted hit@10')
plt.show()

delta_path = Path(result_df.attrs['result_paths']['delta_csv'])
print('M1 대비 paired delta:')
display(pd.read_csv(delta_path).sort_values(['model_id', 'lambda', 'metric']))

screening_decision = result_df.attrs['screening_decision']
print('최종 screening 판정:', screening_decision['success'])
print('사유:', screening_decision['reason'])
print('통과하지 못한 대조군:', screening_decision['failed_controls'])
print('선택 λ:', result_df.attrs['selected_lambda'])
print('결과 파일:')
for label, path in result_df.attrs['result_paths'].items():
    print(f' - {label}: {path}')